# 01.8 — The Python standard library for MATLAB users

A quick tour of the standard library modules this codebase actually uses. The goal isn't to teach every module exhaustively — it's to give you the 80% you need to read `src/neural_data_decoding/` fluently, plus pointers to docs for the other 20%.

Modules covered (in rough order of how often they appear in the codebase):

1. `pathlib` — filesystem paths.
2. `os` — environment variables, process info.
3. `json` — JSON read/write.
4. YAML (via `pyyaml` / `omegaconf` — third-party, but the config format this project lives on).
5. `subprocess` — running shell commands.
6. `datetime` — timestamps and durations.
7. `argparse` — command-line argument parsing.
8. `logging` — structured output (vs ad-hoc `print`).
9. `collections` — `Counter`, `defaultdict`, `namedtuple`.
10. `dataclasses` (already covered in [01.7](01.7_dataclasses_and_typed_configs.ipynb)).

**Prerequisite:** [01.5 modules and imports](01.5_modules_and_imports.ipynb).

## Section 1 — What MATLAB does

MATLAB's standard library is built into the desktop. Filesystem paths are strings: `fullfile('a', 'b', 'c.mat')` joins them, `fileparts(p)` splits them. Environment access is `getenv('FOO')`. There's no concept of "the standard library" because everything ships in one big bag.

Python's standard library is a deliberate, modular collection of 200+ modules. Each does one thing. You import what you need. Anything you don't import doesn't burden your namespace or start-up time.

## Section 2 — `pathlib` — filesystem paths

`pathlib.Path` is the modern way to manipulate filesystem paths. Far cleaner than the old `os.path` functions.

In [ ]:
from pathlib import Path

# Construct a path from parts
p = Path("/tmp") / "data" / "trial_001.mat"
print(p)
print(type(p))    # Path object, not a string

In [ ]:
# Path components
p = Path("/Users/cgerrity/Documents/Projects/neural_data_decoding/configs/base.yaml")
print(p.name)        # base.yaml
print(p.stem)        # base (no extension)
print(p.suffix)      # .yaml
print(p.parent)      # /Users/cgerrity/Documents/Projects/neural_data_decoding/configs
print(p.parts)       # tuple of every component

In [ ]:
# Common operations
p = Path("../..")
print(p.resolve())        # absolute path, symlinks resolved
print(p.exists())          # True / False
print(p.is_file(), p.is_dir())

In [ ]:
# Reading and writing — Path has built-in file methods
src = Path("../../README.md")
content = src.read_text()
print(f"  README is {len(content)} chars")
print(f"  first line: {content.splitlines()[0]}")

In [ ]:
# Globbing — find files matching a pattern
configs = Path("../../configs")
yaml_files = sorted(configs.rglob("*.yaml"))
for f in yaml_files[:5]:
    print(f"  {f.relative_to(configs)}")

**MATLAB ↔ Python path operations:**

| MATLAB | Python (`pathlib.Path`) |
|---|---|
| `fullfile('a', 'b', 'c')` | `Path('a') / 'b' / 'c'` |
| `fileparts(p)` (path, name, ext) | `p.parent`, `p.stem`, `p.suffix` |
| `exist(p, 'file')` | `p.is_file()` |
| `mkdir(p)` | `p.mkdir(parents=True, exist_ok=True)` |
| `fileread(p)` | `p.read_text()` |
| `dir('*.mat')` | `Path('.').glob('*.mat')` |
| `dir(p)` recursive | `Path(p).rglob('*')` |

In our codebase: `interop/folder_hierarchy_matlab.py` chains `Path / "Aggregate Data" / "Epoched Data" / ...` for the MATLAB folder tree. `cli.py` locates its configs directory with `Path(__file__).resolve().parent...` — the standard locate-relative-to-this-module pattern.

## Section 3 — `os` — environment, process info

`os` is the older general-purpose OS interface. For paths, prefer `pathlib`. For env vars and process info, `os` is still the right module.

In [ ]:
import os

# Environment variables — accessor returns str or None
home = os.environ.get("HOME")
user = os.environ.get("USER") or os.environ.get("LOGNAME")
print(f"  HOME: {home}")
print(f"  USER: {user}")

In [ ]:
# Process info
print(f"  PID: {os.getpid()}")
print(f"  CPU count: {os.cpu_count()}")

In our codebase: `sweeps/user_identity.py` reads `$USER` / `$LOGNAME` to detect Charles. `sweeps/slurm_template.py` reads `$SLURM_ARRAY_TASK_ID` (which sbatch sets per task) — though that one lives in the emitted bash script, not in Python.

**`os.environ.get(name, default)` vs `os.environ[name]`:** `get` returns `None` (or your default) for missing vars; `[ ]` raises `KeyError`. Prefer `.get(...)` unless the variable is required.

## Section 4 — `json` — JSON read/write

Python's standard JSON module. Use for any structured data interchange — between Python processes, between Python and TypeScript, for cache files, etc.

In [ ]:
import json

# Encode: Python → JSON string
data = {"name": "GRU", "hidden": [1000, 500, 250], "active": True}
print(json.dumps(data))                # compact
print(json.dumps(data, indent=2))       # pretty

In [ ]:
# Decode: JSON string → Python
js = '{"name": "GRU", "hidden": [1000, 500, 250]}'
parsed = json.loads(js)
print(parsed)
print(parsed["hidden"][-1])             # 250

In [ ]:
# Read / write files
from pathlib import Path
import json

tmp = Path("/tmp/example.json")
tmp.write_text(json.dumps({"x": 1, "y": [1, 2, 3]}, indent=2))
loaded = json.loads(tmp.read_text())
print(loaded)

**JSON ↔ Python type mapping:**

| JSON | Python |
|---|---|
| `{...}` | `dict` |
| `[...]` | `list` |
| `"..."` | `str` |
| `true`/`false` | `True`/`False` |
| `null` | `None` |
| number | `int` or `float` |

The codebase uses `json` for the `check-existing` subcommand's machine-readable output. For human-readable configs, YAML wins (next section).

## Section 5 — YAML via `pyyaml` (or `omegaconf` in this codebase)

YAML is JSON's human-friendly cousin: indentation-based, comments allowed, multi-document, anchors and references. The standard library doesn't include a YAML parser — we use `pyyaml` (the de-facto standard) and `omegaconf` (for Hydra-composable configs).

The codebase's configs are YAML files in `configs/`. Loading them at runtime uses `OmegaConf.load(...)`.

In [ ]:
from omegaconf import OmegaConf

cfg = OmegaConf.load("../../configs/base.yaml")
print(f"  model_name: {cfg.model_name}")
print(f"  hidden_sizes: {cfg.hidden_sizes}")
print(f"  initial_learning_rate: {cfg.initial_learning_rate}")

In [ ]:
# Modify and write back (works in memory; OmegaConf also has save)
cfg.dropout = 0.3
print(f"  dropout (in memory): {cfg.dropout}")

For plain YAML round-trip without Hydra semantics, use `pyyaml` directly:

```python
import yaml

data = yaml.safe_load(open("file.yaml"))
yaml.safe_dump(data, open("out.yaml", "w"))
```

Always use `safe_load` / `safe_dump`, NOT `yaml.load` (which can execute arbitrary code on untrusted input).

## Section 6 — `subprocess` — running shell commands

When you need to invoke an external program — `git`, `matlab`, `nvidia-smi`, anything — use `subprocess.run`.

In [ ]:
import subprocess

# Simple: run, capture output
result = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"],
    capture_output=True,
    text=True,
    check=False,
)
print(f"  returncode: {result.returncode}")
print(f"  stdout: {result.stdout.strip()}")
print(f"  stderr: {result.stderr.strip()!r}")

**Always pass the command as a list, not a string.** `subprocess.run(["git", "status"])` is safe; `subprocess.run("git status", shell=True)` opens a shell injection vector and breaks on filenames with spaces.

**Three important kwargs:**

| Kwarg | Effect |
|---|---|
| `capture_output=True` | collect stdout/stderr into the result object (otherwise they go to the terminal) |
| `text=True` | decode stdout/stderr as text strings (otherwise `bytes`) |
| `check=True` | raise `CalledProcessError` on non-zero exit (vs silently returning the bad code) |

In our codebase: `sweeps/banner.py` and `sweeps/user_identity.py` use `subprocess.run` to call `git config` and `git rev-parse`. Both pass `check=False` and `capture_output=True` so a missing git binary degrades gracefully (returns empty string instead of crashing the banner).

## Section 7 — `datetime` — timestamps and durations

In [ ]:
from datetime import datetime, timezone, timedelta

# Current time
now = datetime.now(tz=timezone.utc)
print(now)
print(now.isoformat())                       # ISO 8601 string

In [ ]:
# Format
print(now.strftime("%Y-%m-%dT%H:%M:%SZ"))     # the banner's format

In [ ]:
# Arithmetic
later = now + timedelta(hours=48)
print(f"  now + 48h: {later.isoformat()}")
print(f"  delta: {later - now}")              # 2 days, 0:00:00

**Always include `tz=timezone.utc`** when you call `datetime.now()`. Naive datetimes (no timezone) cause subtle bugs across machines. The banner module uses UTC ISO 8601 consistently so logs are unambiguous regardless of where they ran.

## Section 8 — `argparse` — command-line argument parsing

The CLI (`src/neural_data_decoding/cli.py`) is built on `argparse`. Worth understanding even if you don't write new CLIs every day.

In [ ]:
import argparse

# Build a parser
parser = argparse.ArgumentParser(description="A toy CLI")
parser.add_argument("--name", type=str, default="world")
parser.add_argument("--count", type=int, default=1)
parser.add_argument("--verbose", action="store_true")

# Parse a hand-rolled argv (instead of sys.argv)
args = parser.parse_args(["--name", "GRU", "--count", "3", "--verbose"])
print(args)
print(f"  args.name = {args.name!r}")

**`action="store_true"` makes a flag** — `--verbose` sets the value to True; absence keeps the default.

**Subcommands** (like `train` / `check-existing` / `sweep-emit-slurm` in our CLI) use `parser.add_subparsers()`. See `src/neural_data_decoding/cli.py:main()` for the full pattern.

## Section 9 — `logging` — structured output

`print()` is fine for one-off debugging. For anything you want to keep, route through the `logging` module — it gives you level filtering (`DEBUG`/`INFO`/`WARNING`/`ERROR`), timestamps, module names, configurable destinations.

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(name)s %(levelname)s | %(message)s",
)

log = logging.getLogger("example")
log.info("starting up")
log.warning("file not found — using default")
log.error("training diverged")

Most of `src/neural_data_decoding/` currently uses `print(...)` for monitoring output (because the training-loop output is the primary user-visible signal, and `print` keeps it simple). If you add structured background work — e.g. a sweep dispatcher that runs unattended on the cluster — `logging` is the better choice.

## Section 10 — `collections` — specialized containers

Three classes from `collections` show up often:

In [ ]:
from collections import Counter, defaultdict

# Counter — tally occurrences
words = ["GRU", "LSTM", "GRU", "Feedforward", "GRU", "LSTM"]
counts = Counter(words)
print(counts)
print(counts.most_common(2))

In [ ]:
# defaultdict — auto-create missing keys
groups = defaultdict(list)
for word in words:
    first = word[0]
    groups[first].append(word)
print(dict(groups))

In [ ]:
# namedtuple — like a frozen dataclass without the import
from collections import namedtuple
Point = namedtuple("Point", ["x", "y"])
p = Point(3, 4)
print(p)
print(p.x, p.y)
# For new code, prefer @dataclass(frozen=True, slots=True) — more features, same speed.

**When to use which:**

| Need | Pick |
|---|---|
| Count occurrences | `Counter` |
| Dict where missing keys default to a fresh container | `defaultdict(list)` / `defaultdict(int)` |
| Immutable record with named fields | `@dataclass(frozen=True, slots=True)` (preferred) or `namedtuple` (legacy) |
| Iterate in insertion order | regular `dict` (Python 3.7+ guarantees this) |

`collections.OrderedDict` is mostly obsolete in modern Python — plain `dict` preserves insertion order since 3.7.

## Section 11 — `dataclasses` (cross-reference)

Already covered in [01.7 dataclasses and typed configs](01.7_dataclasses_and_typed_configs.ipynb). Quick recap:

```python
from dataclasses import dataclass, field

@dataclass(frozen=True, slots=True)
class SweepEntry:
    sweep_index: int
    description: str
    overrides: dict = field(default_factory=dict)
```

The codebase default for new value-like classes.

## Section 12 — The `neural_data_decoding` implementation

Here's a single ~110-line file that uses four of these modules together — `sweeps/user_identity.py`:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/sweeps/user_identity.py").read_text().splitlines()
for i, line in enumerate(src, start=1):
    if i > 108:
        break
    print(f"{i:3} | {line}")

Things to spot:

* `import os` for env vars (`os.environ.get("USER")`).
* `import subprocess` for the `git config` lookup.
* `from dataclasses import dataclass` (the `@dataclass(frozen=True, slots=True)` pattern).
* `from pathlib import Path` for the optional `cwd` argument.
* `frozenset` (a built-in immutable set type — fast hashable lookup).
* Type hints on every function signature.

Four standard-library modules (`os`, `subprocess`, `dataclasses`, `pathlib`) — plus the `frozenset` builtin and type hints — in ~110 lines, each picked for what it does best.

## Section 13 — Hands-on exercises

### Exercise 13.1 — pathlib

Given `p = Path("/scratch/decision_data/Decision_Data_0000011.mat")`, extract the trial ID (`11`) as an int.

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
from pathlib import Path
p = Path("/scratch/decision_data/Decision_Data_0000011.mat")
trial_id = int(p.stem.split("_")[-1])
print(trial_id)

### Exercise 13.2 — datetime

Print the current UTC time in the banner's exact format (`2026-06-04T00:35:52Z`).

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
from datetime import datetime, timezone
print(datetime.now(tz=timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"))

### Exercise 13.3 — Counter

Given the sweep entries' descriptions, find the 3 most-common first words.

In [ ]:
from neural_data_decoding.sweeps import SWEEP_ENTRIES
from collections import Counter

first_words = [e.description.split()[0] for e in SWEEP_ENTRIES]
print(Counter(first_words).most_common(3))

## Section 14 — Common errors

### `FileNotFoundError: [Errno 2] No such file or directory: ...`

The path doesn't exist (typo, wrong CWD, file not committed). Print `Path(p).resolve()` and `Path(p).parent.exists()` to localize.

### `PermissionError: [Errno 13] Permission denied`

Filesystem rights issue. Common when writing to system directories — use `tmp_path` (pytest fixture), `/tmp`, or somewhere under your home.

### `json.JSONDecodeError: Expecting value`

The file isn't valid JSON — often a trailing comma (legal in YAML/Python, not JSON). Single quotes are also illegal in JSON but produce a different message: `Expecting property name enclosed in double quotes`.

### `subprocess.CalledProcessError: Command '[...]' returned non-zero exit status N`

The subprocess failed. With `check=True` this raises; with `check=False` you check `result.returncode` manually.

### `TypeError: Object of type Path is not JSON serializable`

JSON doesn't know about `Path`. Convert to string first: `json.dumps({"path": str(p)})`.

### `AttributeError: 'NoneType' object has no attribute 'group'`

A regex `search()` returned `None` (no match) and you called `.group()` on it. Always check `if match: ...` before unpacking.

## Section 15 — Further reading

- [Python standard library docs](https://docs.python.org/3/library/) — the canonical reference. The TOC alone is worth bookmarking.
- [pathlib docs](https://docs.python.org/3/library/pathlib.html) — every method on `Path`.
- [The Hitchhiker's Guide to Python — Common stdlib](https://docs.python-guide.org/scenarios/) — practical examples.

**Module 01 is complete.** Next up: **[Module 02 — NumPy & PyTorch basics](../02_numpy_and_pytorch_basics/)** (next curriculum batch). Module 02 covers `numpy_vs_matlab_arrays.ipynb`, `array_axis_conventions.ipynb`, `loading_mat_files.ipynb`, `pytorch_tensors_intro.ipynb`, `autograd_basics.ipynb`, `nn_module_vs_layergraph.ipynb`, `optimizers_and_learning_rates.ipynb`, and `nan_handling.ipynb`.